# Thermal state preparation

All 16 source stages, both six-bit Fourier windows with 10,000 samples each, and 50 delta cycles with the original four-bit mixing window and 1,000 thermal samples. All 400 resets execute as channels.

Compute the windows, signed Glauber table and time controls from the original source constants. Use a local sample seed for each circuit. Compare both absolute complex Fourier states and the complete final density matrix, including all reset bits. The Gibbs comparison uses the source tolerance of 0.1.

## Load the native Fourier and thermal workflow

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.algorithms.quantum_state_preparation.gibbs.quantum_thermal_state_preparation.overview.model import (
    BETA,
    DELTA,
    ENERGIES,
    MIXING_BITS,
    ROUNDS,
    SPECTRUM_BITS,
    fourier_spacing,
    glauber_angles,
)
from examples.synthesis.algorithms.quantum_state_preparation.gibbs.quantum_thermal_state_preparation.overview.plots import (
    save_gibbs,
    save_spectra,
    save_window,
)
from examples.synthesis.algorithms.quantum_state_preparation.gibbs.quantum_thermal_state_preparation.overview.workflow import (
    run_workflow,
)

folder = Path("artifacts/development/thermal-state")
print("Hamiltonian eigenvalues:", ENERGIES)


## Generate the Gaussian window

In [ ]:
display(Image(filename=str(save_window(folder / "window.png"))))


## Retain the controlled evolution convention

In [ ]:
print(
    "Prepare a window, conjugate the jump by the energy evolution, then apply inverse QFT."
)


## Inspect the frequency resolutions

In [ ]:
print("Fourier spacings:", fourier_spacing(SPECTRUM_BITS), fourier_spacing(MIXING_BITS))


## Execute the original finite-shot workflow

In [ ]:
result = run_workflow()


## Inspect Gaussian frequency counts

In [ ]:
print("Gaussian spectrum counts:", result.gaussian.counts)


## Compare both measured frequency windows

In [ ]:
print("Uniform spectrum counts:", result.uniform.counts)
display(Image(filename=str(save_spectra(result, folder / "spectra.png"))))


## Keep all four Pauli jumps

In [ ]:
print("Each of the two system wires has an X jump and a Y jump.")


## Regenerate the signed Glauber angle table

In [ ]:
print("Glauber angles:", glauber_angles())


## Execute physical reset in each delta cycle

In [ ]:
print("The two jump bits, four window bits and both flags are reset at each step.")


## Retain the original channel parameters

In [ ]:
print("Beta, delta and rounds:", BETA, DELTA, ROUNDS)


## Read native resources for all fifty cycles

In [ ]:
print("Thermal resources:", result.thermal_program.report["resources"])


## Read actual histogram counts

In [ ]:
print(
    "Actual Gaussian, uniform and thermal shots:",
    sum(result.gaussian.counts),
    sum(result.uniform.counts),
    sum(result.thermal.counts),
)
print("Thermal counts:", result.thermal.counts)


## Normalize the finite sample frequencies

In [ ]:
measured = tuple(count / sum(result.thermal.counts) for count in result.thermal.counts)


## Compare to the independent Gibbs prediction

In [ ]:
print("Gibbs reference:", result.exact_gibbs)


## Check the original Gibbs tolerance

In [ ]:
assert all(
    abs(measurement - prediction) < 0.1
    for measurement, prediction in zip(measured, result.exact_gibbs, strict=True)
)
display(Image(filename=str(save_gibbs(result, folder / "gibbs.png"))))
